# Language models

We'll try talking to some language models:

- An n-gram model
- An untrained transformer
- A base model
- An instruction tuned model

In [1]:
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')
__ = load_dotenv(".env")

---
## An n-gram model of language

In [2]:
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /Users/MTHA/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Train an n-gram model on the Wikipedia page for Equinor.

In [3]:
import requests
from collections import defaultdict, Counter
from nltk.util import ngrams

headers = {"User-Agent": "User:Kwinkunks (kwinkunks@gmail.com)"}
r = requests.get("https://en.wikipedia.org/wiki/Equinor", headers=headers)
tokens = nltk.word_tokenize(r.text)

n = 3
trigrams = list(ngrams(tokens, n))

model = defaultdict(Counter)
for gram in trigrams:
    context = gram[:n-1]
    next_word = gram[n-1]
    model[context][next_word] += 1

for context in model:
    total_count = float(sum(model[context].values()))
    for next_word in model[context]:
        model[context][next_word] /= total_count

model[('Equinor', 'has')]

Counter({'a': 0.2857142857142857,
         'kept': 0.14285714285714285,
         'stakes': 0.14285714285714285,
         'also': 0.14285714285714285,
         'begun': 0.14285714285714285,
         'injected': 0.14285714285714285})

We can draw from this distribution iteratively.

In [11]:
sentence = "Equinor has"

for _ in range(20):
    next_word = model[tuple(sentence.split()[-2:])].most_common(1)[0][0]
    sentence += " " + next_word

sentence

"Equinor has a long history of attempting to get involved in the < a href= '' # '' class= '' cite-bracket ''"

---

## Initial weights: GPT-1 (June 2018)

Let's take a look at GPT-1, one of the first implementations of the Transformer architecture (Vaswani et al, 2017).

Ref: Attention Is All You Need (2017). Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin. https://arxiv.org/abs/1706.03762

We'll load it with random weights -- so this model is completely untrained.

In [16]:
from transformers import OpenAIGPTConfig, OpenAIGPTLMHeadModel, OpenAIGPTTokenizer
import torch

config = OpenAIGPTConfig()
model = OpenAIGPTLMHeadModel(config)
tokenizer = OpenAIGPTTokenizer.from_pretrained('openai-gpt')

inputs = tokenizer("The future of artificial intelligence is", return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=1.0,
        # GPT-1 doesn't have a default pad token, so we can use eos_token if needed
        pad_token_id=tokenizer.eos_token_id 
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

the future of artificial intelligence is delicately delicately merge zona blare blare konev statstirrdened sinful sinful н toasted ancestor instructed ssional frantically dto dto killed routed sephheidi bursar impossibly vials vials tread delicately delicately aires rampant wellwellwellleaders leaders scrambling ously mentioning mentioning addri coax insubstantial 62 62 62 vermin willingness willingness ibohhh honoured festifestiregregoulbunching frail canoe bil inland endaddiction excrupleasuring ining ining ckle bringing bringing awakening clattering fortifortiranger ranger swords


The output makes no sense at all, because the weights are random. you're just seeing a random collection of tokens.

---

## After pre-training: GPT-1 base (June 2018)

In [17]:
from transformers import pipeline, GenerationConfig

gen = pipeline("text-generation", model="openai-gpt")
gen.model.generation_config = GenerationConfig()

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/479M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

OpenAIGPTLMHeadModel LOAD REPORT from: openai-gpt
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

In [18]:
result = gen("The future of artificial intelligence is")
print(result[0]["generated_text"])

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The future of artificial intelligence is something that has been going on for thousands of years. it was created to control us, to control our lives and our futures. it has taken over our minds and bodies. it is a demon that exists only in our bodies. it has taken over our spirit and is constantly within our bodies. " 
 " why? " 
 " because it is a creature that has been imprisoned in our bodies, that has been destroyed. " 
 " why? " 
 " it has been a part of the past. it has been a part of the future. it has helped us to rule, and destroy. it has been a part of the past. it has been the reason why we were created, and why we are here now. it has been a part of the past that must be left behind. it must be destroyed. " 
 " how will you die? " 
 " we do not know. we can not be sure. " 
 " why? i mean, what am i supposed to do? " 
 " you are supposed to be human. " 
 " oh. " 
 " yes. you are supposed to be human. " 
 " i'm not supposed to be human. i'm not supposed to be human. " 
 " yes

---
## After pre-training: GPT-2 base (ca. Feb 2019)

From Wikipedia:

> Documentation surrounding the cost of training GPT-2 is limited. According to a recent statement by Andrej Karpathy, GPT-2 was trained by OpenAI on 32 TPU v3 chips for 168 hours (7 days), at approximately 8 USD per TPU v3 per hour, for a total estimated compute cost of about 43&nbsp;000 USD.

This initial version has about 124 million parameters. It has **not** been instruction tuned.

In [19]:
from transformers import pipeline, GenerationConfig

gen = pipeline("text-generation", model="gpt2")
gen.model.generation_config = GenerationConfig(pad_token_id=50256)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [20]:
result = gen("The future of artificial intelligence is")
print(result[0]["generated_text"])

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The future of artificial intelligence is uncertain. Artificial Intelligence is a disruptive technology. It can never make it to a computer's processor, which is one of the most important goals of the computer industry. It is already well past the point where it is capable of making a real computer, but it will only become a reality if it can demonstrate a human understanding of the underlying problem. It is not clear if artificial intelligence will make it to the computer's processor, but it's certainly a very important step in the right direction.

This is important because the future of artificial intelligence is uncertain. Artificial Intelligence is a disruptive technology. It can never make it to a computer's processor, which is one of the most important goals of the computer industry. It is already well past the point where it is capable of making a real computer, but it will only become a reality if it can demonstrate a human understanding of the underlying problem. It is not cle

---

## A later base model: Llama 2 7B (April 2024) and/or Llama 3.2 1B (Nov 2024)

This is Meta's open weight model, and it's about 5 years later than GPT-2 (above).

This model has 7 billion parameters, so more than 50x GPT-2.

This version is still just the base model, so after pretraining but before fine-tuning. Later, we'll see the intruction tuned model.

In [21]:
gen = pipeline("text-generation", model="meta-llama/Llama-2-7b-hf")
# gen = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [22]:
result = gen("The future of artificial intelligence is", max_new_tokens=150)
print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The future of artificial intelligence is all about the human touch.
In fact, it is the only way that AI will ever be able to become mainstream.
In this episode of the WSJ’s “The Future of Everything,” we talk to the founder of Google, Eric Schmidt, about how AI is changing our world.
In the episode, we talk with Eric Schmidt about his vision for the future of AI, and how he plans to make it more accessible to the average person.
The episode is a great introduction to the world of artificial intelligence, and we’re excited to share it with you.
The Future of Everything is produced by The Wall Street Journal and is available on Apple Podcasts, Google Play, Spotify


### Asking questions of a base model is not usually that great

In [23]:
prompt = "Who is the prime minister of the UK?"

result = gen(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Who is the prime minister of the UK?
What are the 3 main political parties in the UK?
What is the most popular political party in the UK?
What is the UK political system?
Who is the prime minister of England?
What is the name of the British Prime Minister?
Who is the youngest Prime Minister of the UK?
What are the 5 political parties in the UK?
What is the most popular political party in the UK?
How many political parties are there in UK?



In [24]:
prompt = "Q: When did the Devonian end?\nA:"

result = gen(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: When did the Devonian end?
A: The Devonian ended about 360 million years ago.
Q: What is the name of the Devonian period?
A: The Devonian period is one of the geologic periods of the Paleozoic Era.
Q: What does the Devonian period mean?
A: The Devonian period is a geologic period that is defined as being between 416 million years ago and 358.9 million years ago.
Q:


---
## Instruction tuned model: Llama 2 7B Chat (April 2024)

This version has been fine-tuned on question-answer pairs.

In [25]:
gen = pipeline("text-generation", model="meta-llama/Llama-2-7b-chat-hf")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [26]:
prompt = "Who is the prime minister of the UK?"

result = gen(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Who is the prime minister of the UK?

The Prime Minister of the United Kingdom is Boris Johnson. He has been in office since July 2019 and is the leader of the Conservative Party. The Prime Minister is the head of the government in the UK and is responsible for setting the policy and direction of the government. They are also the leader of the House of Commons and are responsible for advising the Queen on the appointment of other ministers.


In [27]:
prompt = "When did the Devonian end?"

result = gen(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


When did the Devonian end?

The Devonian period ended about 375 million years ago, during a time of intense volcanic activity and sea-level changes. The Devonian period lasted from 416 to 359 million years ago, and it is known as the "Age of Fishes" because of the abundance of fish species that evolved during this time.


### Strange results?

If the results are strange, you can try using some special tokens the model was trained to recognize as marking the human side of the chat:

- `<s></s>`: These are the BOS and EOS tokens from SentencePiece. When multiple messages are present in a multi turn conversation, they separate them, including the user input and model response.
- `[INST][/INST]`: These tokens enclose user messages in multi turn conversations.
- `<<SYS>><</SYS>>`: These enclose the system message.

In [28]:
prompt = """<s>[INST] Complete this sentence thoughtfully: The future of artificial intelligence is [/INST]"""

result = gen(prompt, max_new_tokens=100)
print(result[0]["generated_text"])

Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s>[INST] Complete this sentence thoughtfully: The future of artificial intelligence is [/INST]  The future of artificial intelligence is _______________ (fill in the blank). There are many possible answers to this question, and it ultimately depends on various factors such as technological advancements, societal needs, and ethical considerations. Here are some possible ways to complete the sentence:

1. The future of artificial intelligence is transformative: AI will continue to revolutionize industries and aspects of our lives, from healthcare and education to transportation and entertainment


---

Matt Hall 2026, licensed CC BY